# 🍎🥦 Fruit & Vegetable Detection — YOLOv8 Hard Sample Mining Pipeline

**Pipeline:** `YOLO (fine-tuned) → Detect hard samples → Hard sample buffer → Weighted re-train (×3 iterations)`

**Dataset:** [LVIS Fruits and Vegetables](https://www.kaggle.com/datasets/henningheyen/lvis-fruits-and-vegetables-dataset)

> ⚠️ **Before running:** Go to `Runtime → Change runtime type → GPU (T4 or better)`

## 📦 Cell 1 — Install Dependencies

In [ ]:
# ============================================================
# CELL 1: Install all required libraries
# ============================================================

!pip install ultralytics kaggle opencv-python matplotlib seaborn Pillow PyYAML --quiet
!pip install torch torchvision --quiet  # Usually pre-installed in Colab

import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "⚠️ No GPU detected — switch to GPU runtime in Colab!")

## 🔑 Cell 2 — Kaggle API Setup

In [ ]:
# ============================================================
# CELL 2: Configure Kaggle API credentials
# Upload your kaggle.json from https://www.kaggle.com/settings
# ============================================================

import os
from google.colab import files

print("📂 Please upload your kaggle.json file...")
uploaded = files.upload()

# Place credentials in the correct directory
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json
print("✅ Kaggle API credentials configured successfully.")

## 📥 Cell 3 — Download & Organize Dataset

In [ ]:
# ============================================================
# CELL 3: Download dataset from Kaggle and organize in YOLO format
# ============================================================

import os
import shutil
import zipfile
import glob
import random

# ── Configuration ──────────────────────────────────────────
BASE_DIR       = '/content/fruits_veg'
DATASET_DIR    = os.path.join(BASE_DIR, 'dataset')
IMAGES_TRAIN   = os.path.join(DATASET_DIR, 'images/train')
IMAGES_VAL     = os.path.join(DATASET_DIR, 'images/val')
LABELS_TRAIN   = os.path.join(DATASET_DIR, 'labels/train')
LABELS_VAL     = os.path.join(DATASET_DIR, 'labels/val')
HARD_IMG_DIR   = os.path.join(BASE_DIR, 'hard_samples/images')
HARD_LBL_DIR   = os.path.join(BASE_DIR, 'hard_samples/labels')
MODELS_DIR     = os.path.join(BASE_DIR, 'models')
DOWNLOAD_DIR   = '/content/kaggle_download'
VAL_SPLIT      = 0.15   # 15% for validation
RANDOM_SEED    = 42
# ───────────────────────────────────────────────────────────

def create_directories():
    """Create all required directory structure."""
    dirs = [
        IMAGES_TRAIN, IMAGES_VAL, LABELS_TRAIN, LABELS_VAL,
        HARD_IMG_DIR, HARD_LBL_DIR, MODELS_DIR, DOWNLOAD_DIR
    ]
    for d in dirs:
        os.makedirs(d, exist_ok=True)
    print("✅ Directory structure created.")

def download_dataset():
    """Download the Kaggle dataset."""
    print("📥 Downloading dataset from Kaggle...")
    os.system(f'kaggle datasets download -d henningheyen/lvis-fruits-and-vegetables-dataset -p {DOWNLOAD_DIR}')
    zip_files = glob.glob(os.path.join(DOWNLOAD_DIR, '*.zip'))
    if not zip_files:
        raise FileNotFoundError("❌ Download failed — no zip file found. Check your kaggle.json and quota.")
    print(f"✅ Downloaded: {zip_files[0]}")
    return zip_files[0]

def unzip_dataset(zip_path):
    """Unzip the downloaded dataset."""
    extract_dir = os.path.join(DOWNLOAD_DIR, 'extracted')
    os.makedirs(extract_dir, exist_ok=True)
    print(f"📦 Unzipping {zip_path} ...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_dir)
    print("✅ Unzipped successfully.")
    return extract_dir

def find_yolo_files(root_dir):
    """
    Recursively find all image files and their corresponding YOLO label files.
    Returns list of (image_path, label_path) tuples where both exist.
    """
    image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    pairs = []
    for dirpath, _, filenames in os.walk(root_dir):
        for fname in filenames:
            ext = os.path.splitext(fname)[1].lower()
            if ext in image_exts:
                img_path = os.path.join(dirpath, fname)
                # YOLO label: same name, .txt extension
                lbl_path = os.path.splitext(img_path)[0] + '.txt'
                if os.path.exists(lbl_path):
                    pairs.append((img_path, lbl_path))
    print(f"✅ Found {len(pairs)} image-label pairs.")
    return pairs

def split_and_copy(pairs, val_split=VAL_SPLIT, seed=RANDOM_SEED):
    """Split pairs into train/val and copy into YOLO directory structure."""
    random.seed(seed)
    random.shuffle(pairs)
    n_val = max(1, int(len(pairs) * val_split))
    val_pairs   = pairs[:n_val]
    train_pairs = pairs[n_val:]
    print(f"📊 Split: {len(train_pairs)} train | {len(val_pairs)} val")

    def copy_pairs(pair_list, img_dst, lbl_dst):
        for img_src, lbl_src in pair_list:
            shutil.copy(img_src, os.path.join(img_dst, os.path.basename(img_src)))
            shutil.copy(lbl_src, os.path.join(lbl_dst, os.path.basename(lbl_src)))

    copy_pairs(train_pairs, IMAGES_TRAIN, LABELS_TRAIN)
    copy_pairs(val_pairs,   IMAGES_VAL,   LABELS_VAL)
    print("✅ Files copied into YOLO directory structure.")
    return train_pairs, val_pairs

def extract_class_names(extracted_dir):
    """
    Try to read class names from a YAML / names file in the dataset.
    Falls back to scanning label files to count unique class IDs.
    """
    import yaml
    # Search for YAML config
    yaml_files = glob.glob(os.path.join(extracted_dir, '**/*.yaml'), recursive=True) + \
                 glob.glob(os.path.join(extracted_dir, '**/*.yml'),  recursive=True)
    for yf in yaml_files:
        try:
            with open(yf) as f:
                cfg = yaml.safe_load(f)
            if isinstance(cfg, dict) and 'names' in cfg:
                names = cfg['names']
                if isinstance(names, list) and len(names) > 0:
                    print(f"✅ Loaded {len(names)} class names from {yf}")
                    return names
        except Exception:
            pass

    # Fallback: scan all label files for max class ID
    print("⚠️ No YAML with class names found — inferring from label files...")
    max_cls = 0
    for dirpath, _, filenames in os.walk(extracted_dir):
        for fname in filenames:
            if fname.endswith('.txt'):
                try:
                    with open(os.path.join(dirpath, fname)) as f:
                        for line in f:
                            parts = line.strip().split()
                            if parts:
                                max_cls = max(max_cls, int(parts[0]))
                except Exception:
                    pass
    names = [f"class_{i}" for i in range(max_cls + 1)]
    print(f"⚠️ Inferred {len(names)} classes from label IDs (no names available).")
    return names

def write_yaml(class_names):
    """Write the YOLO dataset YAML configuration file."""
    import yaml
    cfg = {
        'path': DATASET_DIR,
        'train': 'images/train',
        'val':   'images/val',
        'nc':    len(class_names),
        'names': class_names
    }
    yaml_path = os.path.join(DATASET_DIR, 'dataset.yaml')
    with open(yaml_path, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)
    print(f"✅ YOLO YAML written to: {yaml_path}")
    return yaml_path

# ── Run setup ──────────────────────────────────────────────
create_directories()
zip_path     = download_dataset()
extracted    = unzip_dataset(zip_path)
pairs        = find_yolo_files(extracted)
train_pairs, val_pairs = split_and_copy(pairs)
class_names  = extract_class_names(extracted)
YAML_PATH    = write_yaml(class_names)
NUM_CLASSES  = len(class_names)

print(f"\n📋 Dataset Summary:")
print(f"   Classes : {NUM_CLASSES}")
print(f"   Train   : {len(train_pairs)} images")
print(f"   Val     : {len(val_pairs)} images")
print(f"   YAML    : {YAML_PATH}")

## 🛠️ Cell 4 — Helper / Utility Functions

In [ ]:
# ============================================================
# CELL 4: Helper / utility functions used across all cells
# ============================================================

import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from ultralytics import YOLO
from pathlib import Path

def compute_iou(box1, box2):
    """
    Compute IoU between two boxes in [x1,y1,x2,y2] format.
    Both boxes should be absolute pixel coordinates.
    """
    x1 = max(box1[0], box2[0]); y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2]); y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0

def xywh_to_xyxy(cx, cy, w, h, img_w, img_h):
    """Convert YOLO normalized [cx,cy,w,h] to absolute [x1,y1,x2,y2]."""
    x1 = (cx - w/2) * img_w
    y1 = (cy - h/2) * img_h
    x2 = (cx + w/2) * img_w
    y2 = (cy + h/2) * img_h
    return [x1, y1, x2, y2]

def load_ground_truth(label_path, img_w, img_h):
    """Load YOLO format ground-truth labels into list of (class_id, [x1,y1,x2,y2])."""
    gt = []
    if not os.path.exists(label_path):
        return gt
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                cls = int(parts[0])
                cx, cy, w, h = map(float, parts[1:])
                gt.append((cls, xywh_to_xyxy(cx, cy, w, h, img_w, img_h)))
    return gt

def is_hard_sample(results, label_path, conf_thresh=0.5, iou_thresh=0.5):
    """
    Determine whether an image is a hard sample based on:
      1. Any detection has confidence < conf_thresh
      2. Any detection has IoU with ground-truth < iou_thresh
      3. Any detection has wrong class (class mismatch despite high IoU)
    Returns True if image should be added to hard-sample buffer.
    """
    result = results[0]
    img_h, img_w = result.orig_shape[:2]
    gt_boxes = load_ground_truth(label_path, img_w, img_h)

    # No ground-truth file → treat as hard if no detections (missed scene)
    if not gt_boxes and len(result.boxes) == 0:
        return False
    if not gt_boxes:
        return True  # Predicted something when nothing expected

    preds = result.boxes
    if len(preds) == 0:
        return True  # Missed all objects

    hard = False
    for box in preds:
        conf  = float(box.conf[0])
        cls   = int(box.cls[0])
        xyxy  = box.xyxy[0].tolist()

        # Criterion 1: low confidence
        if conf < conf_thresh:
            hard = True
            break

        # Criterion 2 & 3: check against ground-truth
        best_iou  = 0.0
        best_cls  = -1
        for gt_cls, gt_box in gt_boxes:
            iou = compute_iou(xyxy, gt_box)
            if iou > best_iou:
                best_iou = iou
                best_cls = gt_cls

        if best_iou < iou_thresh:      # Localisation error
            hard = True; break
        if best_cls != cls:            # Class error
            hard = True; break

    return hard

def visualize_hard_samples(n=6):
    """Display a grid of hard sample images with their filenames."""
    img_files = sorted(glob.glob(os.path.join(HARD_IMG_DIR, '*')))[:n]
    if not img_files:
        print("⚠️ No hard samples found to visualise.")
        return
    cols = 3; rows = (len(img_files) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
    axes = axes.flatten() if rows > 1 else ([axes] if cols == 1 else axes.flatten())
    for ax, fpath in zip(axes, img_files):
        img = cv2.cvtColor(cv2.imread(fpath), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(os.path.basename(fpath), fontsize=8)
        ax.axis('off')
    for ax in axes[len(img_files):]:
        ax.axis('off')
    plt.suptitle(f"Hard Samples (showing {len(img_files)})", fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()
    print(f"📊 Total hard samples in buffer: {len(glob.glob(os.path.join(HARD_IMG_DIR, '*')))}")

print("✅ Helper functions loaded.")

## 🚀 Cell 5 — Initial Training (Iteration 0)

In [ ]:
# ============================================================
# CELL 5: Initial YOLOv8 training on the base dataset
# ============================================================

def train_yolo(
    model_path,
    yaml_path,
    run_name,
    epochs=20,
    imgsz=640,
    batch=16,
    patience=10
):
    """
    Train a YOLOv8 model. Returns path to the best saved weights.
    model_path : pretrained weights or previous best.pt
    yaml_path  : path to dataset YAML
    run_name   : Ultralytics project run name for saving
    """
    print(f"\n🏋️  Starting training: {run_name}")
    print(f"   Model  : {model_path}")
    print(f"   Epochs : {epochs}  |  Batch : {batch}  |  Imgsz : {imgsz}")

    model = YOLO(model_path)
    model.train(
        data      = yaml_path,
        epochs    = epochs,
        imgsz     = imgsz,
        batch     = batch,
        name      = run_name,
        project   = MODELS_DIR,
        patience  = patience,
        device    = 0 if torch.cuda.is_available() else 'cpu',
        save      = True,
        plots     = True,
        verbose   = True,
        workers   = 2,
    )
    best_weights = os.path.join(MODELS_DIR, run_name, 'weights', 'best.pt')
    if not os.path.exists(best_weights):
        # Fallback to last.pt if best not saved
        best_weights = os.path.join(MODELS_DIR, run_name, 'weights', 'last.pt')
    print(f"✅ Training complete. Best weights: {best_weights}")
    return best_weights

# ── Run initial training ───────────────────────────────────
INITIAL_WEIGHTS = train_yolo(
    model_path = 'yolov8x.pt',   # Pretrained COCO weights (auto-downloaded)
    yaml_path  = YAML_PATH,
    run_name   = 'iter_0_base',
    epochs     = 20,
    batch      = 16,
)
print(f"\n📌 Initial model saved at: {INITIAL_WEIGHTS}")

## 🔍 Cell 6 — Hard Sample Detection

In [ ]:
# ============================================================
# CELL 6: Run inference on validation set → find hard samples
# ============================================================

def detect_hard_samples(
    weights_path,
    conf_thresh = 0.5,
    iou_thresh  = 0.5,
    max_buffer  = 500,
):
    """
    Run YOLOv8 inference on the validation set.
    Identify hard samples (low conf or bad IoU/class) and copy
    them into HARD_IMG_DIR / HARD_LBL_DIR, avoiding duplicates.
    Returns the count of new hard samples added.
    """
    print(f"\n🔍 Detecting hard samples using: {weights_path}")
    model = YOLO(weights_path)

    val_images = sorted(glob.glob(os.path.join(IMAGES_VAL, '*')))
    print(f"   Evaluating {len(val_images)} validation images...")

    # Existing buffer: filenames already saved (to avoid duplicates)
    existing = set(os.listdir(HARD_IMG_DIR))
    added = 0; skipped_dup = 0; buffer_full = False

    for img_path in val_images:
        fname    = os.path.basename(img_path)
        lbl_path = os.path.join(LABELS_VAL, os.path.splitext(fname)[0] + '.txt')

        # Check buffer capacity before inference
        current_buf = len(os.listdir(HARD_IMG_DIR))
        if current_buf >= max_buffer:
            buffer_full = True
            break

        # Duplicate guard
        if fname in existing:
            skipped_dup += 1
            continue

        # Run inference (single image, no save)
        results = model(img_path, verbose=False)

        if is_hard_sample(results, lbl_path, conf_thresh, iou_thresh):
            # Copy image
            shutil.copy(img_path, os.path.join(HARD_IMG_DIR, fname))
            # Copy label if it exists
            if os.path.exists(lbl_path):
                shutil.copy(lbl_path, os.path.join(HARD_LBL_DIR, os.path.splitext(fname)[0] + '.txt'))
            existing.add(fname)
            added += 1

    total_buf = len(os.listdir(HARD_IMG_DIR))
    if buffer_full:
        print(f"⚠️  Buffer full ({max_buffer} images). Stopped early.")
    print(f"✅ Hard sample detection complete.")
    print(f"   New samples added : {added}")
    print(f"   Duplicates skipped: {skipped_dup}")
    print(f"   Total buffer size : {total_buf}")
    return added

# ── Detect hard samples after initial training ─────────────
n_hard = detect_hard_samples(
    weights_path = INITIAL_WEIGHTS,
    conf_thresh  = 0.5,
    iou_thresh   = 0.5,
    max_buffer   = 500,
)

# Visualise a few hard samples
visualize_hard_samples(n=6)

## 🔁 Cell 7 — Build Augmented (Weighted) Dataset

In [ ]:
# ============================================================
# CELL 7: Build augmented dataset (original + oversampled hard samples)
# and write a new YAML for weighted retraining
# ============================================================

import yaml

AUGMENTED_DIR        = os.path.join(BASE_DIR, 'augmented_dataset')
AUG_IMAGES_TRAIN     = os.path.join(AUGMENTED_DIR, 'images/train')
AUG_IMAGES_VAL       = os.path.join(AUGMENTED_DIR, 'images/val')
AUG_LABELS_TRAIN     = os.path.join(AUGMENTED_DIR, 'labels/train')
AUG_LABELS_VAL       = os.path.join(AUGMENTED_DIR, 'labels/val')

def build_augmented_dataset(oversample_factor=3):
    """
    Merge original training data with oversampled hard samples.
    Hard samples are copied `oversample_factor` times (with renamed files)
    to increase their weight during training.
    Val set remains the original validation data (no leakage).

    oversample_factor : how many extra copies of each hard sample to include
    """
    print(f"\n🔧 Building augmented dataset (oversample x{oversample_factor})...")

    # Fresh augmented dirs
    for d in [AUG_IMAGES_TRAIN, AUG_IMAGES_VAL, AUG_LABELS_TRAIN, AUG_LABELS_VAL]:
        if os.path.exists(d):
            shutil.rmtree(d)
        os.makedirs(d)

    # 1. Copy original train data
    orig_imgs = glob.glob(os.path.join(IMAGES_TRAIN, '*'))
    for img_path in orig_imgs:
        fname = os.path.basename(img_path)
        shutil.copy(img_path, os.path.join(AUG_IMAGES_TRAIN, fname))
        lbl_path = os.path.join(LABELS_TRAIN, os.path.splitext(fname)[0] + '.txt')
        if os.path.exists(lbl_path):
            shutil.copy(lbl_path, os.path.join(AUG_LABELS_TRAIN, os.path.splitext(fname)[0] + '.txt'))
    print(f"   Original train images copied : {len(orig_imgs)}")

    # 2. Oversample hard samples into the train split
    hard_imgs = glob.glob(os.path.join(HARD_IMG_DIR, '*'))
    added_hard = 0
    for copy_idx in range(oversample_factor):
        for img_path in hard_imgs:
            fname    = os.path.basename(img_path)
            stem     = os.path.splitext(fname)[0]
            ext      = os.path.splitext(fname)[1]
            new_name = f"hard_{copy_idx}_{fname}"        # unique name per copy
            shutil.copy(img_path, os.path.join(AUG_IMAGES_TRAIN, new_name))
            lbl_path = os.path.join(HARD_LBL_DIR, stem + '.txt')
            if os.path.exists(lbl_path):
                shutil.copy(lbl_path, os.path.join(AUG_LABELS_TRAIN, f"hard_{copy_idx}_{stem}.txt"))
            added_hard += 1
    print(f"   Hard sample copies added     : {added_hard}  ({len(hard_imgs)} unique x {oversample_factor})")

    # 3. Copy original val data (unchanged)
    val_imgs = glob.glob(os.path.join(IMAGES_VAL, '*'))
    for img_path in val_imgs:
        fname = os.path.basename(img_path)
        shutil.copy(img_path, os.path.join(AUG_IMAGES_VAL, fname))
        lbl_path = os.path.join(LABELS_VAL, os.path.splitext(fname)[0] + '.txt')
        if os.path.exists(lbl_path):
            shutil.copy(lbl_path, os.path.join(AUG_LABELS_VAL, os.path.splitext(fname)[0] + '.txt'))
    print(f"   Val images copied            : {len(val_imgs)}")

    # 4. Write augmented dataset YAML
    aug_yaml_path = os.path.join(AUGMENTED_DIR, 'dataset.yaml')
    cfg = {
        'path' : AUGMENTED_DIR,
        'train': 'images/train',
        'val'  : 'images/val',
        'nc'   : NUM_CLASSES,
        'names': class_names,
    }
    with open(aug_yaml_path, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)

    total_train = len(glob.glob(os.path.join(AUG_IMAGES_TRAIN, '*')))
    print(f"✅ Augmented dataset ready.")
    print(f"   Total train images (with oversampling): {total_train}")
    print(f"   YAML: {aug_yaml_path}")
    return aug_yaml_path

AUG_YAML_PATH = build_augmented_dataset(oversample_factor=3)
print(f"\n📌 Augmented YAML: {AUG_YAML_PATH}")

## 🔄 Cell 8 — Full Iterative Training Loop (3 Iterations)

In [ ]:
# ============================================================
# CELL 8: Full iterative pipeline
#   iter 1: train on augmented data → detect hard → rebuild
#   iter 2: retrain → detect hard → rebuild
#   iter 3: final retrain
# ============================================================

def run_iteration(iteration, weights_path, yaml_path, epochs=15, oversample=3):
    """
    Execute one full iteration of:
      Train → Detect Hard Samples → Build Augmented Dataset
    Returns (new_weights_path, aug_yaml_path)
    """
    print(f"\n{'='*60}")
    print(f"🔄  ITERATION {iteration}")
    print(f"{'='*60}")

    # ── Step 1: Train ──────────────────────────────────────
    run_name = f"iter_{iteration}_retrain"
    new_weights = train_yolo(
        model_path = weights_path,
        yaml_path  = yaml_path,
        run_name   = run_name,
        epochs     = epochs,
        batch      = 16,
    )

    # ── Step 2: Detect hard samples ────────────────────────
    detect_hard_samples(
        weights_path = new_weights,
        conf_thresh  = 0.5,
        iou_thresh   = 0.5,
        max_buffer   = 500,
    )

    # ── Step 3: Rebuild augmented dataset ──────────────────
    new_yaml = build_augmented_dataset(oversample_factor=oversample)

    print(f"\n✅ Iteration {iteration} complete.")
    print(f"   Weights : {new_weights}")
    print(f"   YAML    : {new_yaml}")
    return new_weights, new_yaml

# ── Tracking metrics across iterations ────────────────────
iteration_log = []

# Iteration 1
weights_iter1, yaml_iter1 = run_iteration(
    iteration    = 1,
    weights_path = INITIAL_WEIGHTS,
    yaml_path    = AUG_YAML_PATH,
    epochs       = 15,
    oversample   = 3,
)
iteration_log.append({'iter': 1, 'weights': weights_iter1})

# Iteration 2
weights_iter2, yaml_iter2 = run_iteration(
    iteration    = 2,
    weights_path = weights_iter1,
    yaml_path    = yaml_iter1,
    epochs       = 15,
    oversample   = 3,
)
iteration_log.append({'iter': 2, 'weights': weights_iter2})

# Iteration 3 (final — train on enriched data, no further rebuild needed)
print(f"\n{'='*60}")
print(f"🔄  ITERATION 3 (Final Training)")
print(f"{'='*60}")
FINAL_WEIGHTS = train_yolo(
    model_path = weights_iter2,
    yaml_path  = yaml_iter2,
    run_name   = 'iter_3_final',
    epochs     = 15,
    batch      = 16,
)
iteration_log.append({'iter': 3, 'weights': FINAL_WEIGHTS})

print(f"\n🏆  All 3 iterations complete!")
for entry in iteration_log:
    print(f"   Iter {entry['iter']}: {entry['weights']}")

## 💾 Cell 9 — Save Final Model

In [ ]:
# ============================================================
# CELL 9: Save the final model with the required filename
# ============================================================

FINAL_MODEL_PATH = '/content/yolo_fruits_and_vegetables.pt'

shutil.copy(FINAL_WEIGHTS, FINAL_MODEL_PATH)
print(f"✅ Final model saved as: {FINAL_MODEL_PATH}")

# Quick verification
model_check = YOLO(FINAL_MODEL_PATH)
print(f"   Model class count : {model_check.model.nc}")
print(f"   Model class names : {model_check.names}")

## 📊 Cell 10 — Before vs After Evaluation & Visualization

In [ ]:
# ============================================================
# CELL 10: Compare initial vs final model performance
# Visualise hard samples + inference examples
# ============================================================

def evaluate_model(weights_path, yaml_path, label):
    """Run YOLO val() and return mAP50 and mAP50-95."""
    model  = YOLO(weights_path)
    device = 0 if torch.cuda.is_available() else 'cpu'
    metrics = model.val(data=yaml_path, device=device, verbose=False)
    map50    = metrics.box.map50
    map5095  = metrics.box.map
    print(f"  [{label}]  mAP@50: {map50:.4f}  |  mAP@50:95: {map5095:.4f}")
    return map50, map5095

print("\n📊 Evaluating models (this may take a few minutes)...")

results_log = {}
for entry in [{'iter': 0, 'weights': INITIAL_WEIGHTS}] + iteration_log:
    label = f"Iter {entry['iter']}"
    m50, m5095 = evaluate_model(entry['weights'], YAML_PATH, label)
    results_log[label] = {'mAP50': m50, 'mAP50_95': m5095}

# ── Plot before vs after ───────────────────────────────────
labels = list(results_log.keys())
map50_vals   = [results_log[k]['mAP50']    for k in labels]
map5095_vals = [results_log[k]['mAP50_95'] for k in labels]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, map50_vals,   width, label='mAP@50',    color='steelblue')
bars2 = ax.bar(x + width/2, map5095_vals, width, label='mAP@50:95', color='darkorange')

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Training Iteration')
ax.set_ylabel('mAP Score')
ax.set_title('Before vs After: mAP Improvement Across Iterations')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.legend(); ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout(); plt.show()

# ── Visualise hard samples ─────────────────────────────────
print("\n🖼️  Hard samples in buffer:")
visualize_hard_samples(n=9)

# ── Run inference on a few val images with the final model ─
def show_inference(weights_path, n=4):
    """Show final model inference on random val images."""
    model     = YOLO(weights_path)
    val_imgs  = random.sample(sorted(glob.glob(os.path.join(IMAGES_VAL, '*'))), min(n, 4))
    fig, axes = plt.subplots(1, len(val_imgs), figsize=(5 * len(val_imgs), 5))
    if len(val_imgs) == 1:
        axes = [axes]
    for ax, img_path in zip(axes, val_imgs):
        results = model(img_path, verbose=False)
        annotated = results[0].plot()          # BGR numpy array with boxes drawn
        annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        ax.imshow(annotated)
        ax.set_title(os.path.basename(img_path), fontsize=8)
        ax.axis('off')
    plt.suptitle("Final Model Inference on Validation Images", fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

print("\n🔭 Final model inference examples:")
show_inference(FINAL_MODEL_PATH, n=4)

print(f"\n{'='*60}")
print(f"🏁  Pipeline Complete!")
print(f"    Final model : {FINAL_MODEL_PATH}")
print(f"    Hard buffer : {len(glob.glob(os.path.join(HARD_IMG_DIR, '*')))} samples")
print(f"{'='*60}")